# Remote Brain C Orchestrator Simulation

CPU-only Phase 2 simulation for the updated Three-Brain architecture. Brain A and Brain B are local. Brain C is represented by the same task-shaped wrappers used by the Go service, with optional remote mode through `BRAIN_C_URL`.

In [ ]:
import json, math, os, re, sqlite3, statistics, time, urllib.request

LEVEL_NAME = {1: 'junior', 2: 'mid', 3: 'senior', 4: 'senior'}
FILLER_RE = re.compile(r'<filler>(.*?)</filler>', re.S | re.I)
RESPONSE_RE = re.compile(r'<response>(.*?)</response>', re.S | re.I)
SCORE_RE = re.compile(r'Score:\s*(\d+(?:\.\d+)?)\s*/\s*10', re.I)

def wrap_generate(topic, level, tags=None):
    tags_clause = '' if not tags else ' Relevant tags: ' + ', '.join(tags) + '.'
    return f"Generate a {LEVEL_NAME.get(level, 'mid')}-level interview question about: {topic}.{tags_clause}"

def wrap_evaluate(question, answer):
    return f"Evaluate this candidate answer.\n\nQuestion: {question.strip()}\n\nCandidate answer: {answer.strip()}"

def wrap_rephrase(question):
    return 'Rephrase the following interview question while preserving its intent:\n\n' + question.strip()

def parse_reply(raw):
    filler = FILLER_RE.search(raw)
    response = RESPONSE_RE.search(raw)
    score = SCORE_RE.search(raw)
    return {
        'filler': filler.group(1).strip() if filler else '',
        'response': response.group(1).strip() if response else raw.strip(),
        'score_0_to_1': float(score.group(1)) / 10 if score else None,
        'raw': raw,
    }


In [ ]:
def embed(text, dims=64):
    vec = [0.0] * dims
    for tok in text.lower().split():
        vec[hash(tok) % dims] += 1.0
    norm = math.sqrt(sum(v*v for v in vec)) or 1.0
    return [v / norm for v in vec]

class BrainA:
    def __init__(self):
        self.db = sqlite3.connect(':memory:')
        self.db.execute('create table turns(text text, language text, vec text)')
    def store(self, text, language='auto'):
        self.db.execute('insert into turns values (?, ?, ?)', (text, language, ','.join(map(str, embed(text)))))
        self.db.commit()
    def recall(self, query, k=2):
        q = embed(query)
        scored = []
        for text, _, raw in self.db.execute('select text, language, vec from turns'):
            vec = [float(x) for x in raw.split(',')]
            scored.append((sum(a*b for a, b in zip(q, vec)), text))
        return [text for _, text in sorted(scored, reverse=True)[:k]]

class BrainB:
    def analyze(self, text):
        lower = text.lower()
        hits = sum(m in lower for m in ['maine', 'mujhe', 'nhi', 'nahi', 'kyunki', 'ke baad', 'samajh'])
        language = 'hinglish' if hits else 'en'
        if re.search(r'[\u0900-\u097F]', text): language = 'hi'
        directive = 'REPHRASE' if any(x in lower for x in ['not sure', 'clear nahi', 'nhi ata']) else 'NEUTRAL'
        ack = {'en': 'That makes sense.', 'hi': 'Haan, samajh raha hoon.', 'hinglish': 'Got it, yeh makes sense.'}[language]
        return {'directive': directive, 'language': language, 'ack': ack}


In [ ]:
class BrainCClient:
    def __init__(self):
        self.base_url = os.environ.get('BRAIN_C_URL', '').rstrip('/')
        self.mode = 'remote' if self.base_url else 'mock'
    def chat(self, messages, max_tokens=300, temperature=0.6):
        if not self.base_url:
            prompt = messages[-1]['content']
            if prompt.startswith('Evaluate this candidate answer'):
                return '<filler>Got it.</filler><response>Score: 7/10. Concrete answer with room for clearer tradeoff reasoning.</response>'
            if prompt.startswith('Rephrase'):
                return '<response>Let me ask it more simply: what signal would you check first?</response>'
            return '<filler>Alright.</filler><response>Can you walk me through one concrete tradeoff and the metric that proved it worked?</response>'
        payload = json.dumps({'model': 'customllm', 'messages': messages, 'max_tokens': max_tokens, 'temperature': temperature}).encode()
        req = urllib.request.Request(self.base_url + '/v1/chat/completions', data=payload, headers={'Content-Type': 'application/json'})
        with urllib.request.urlopen(req, timeout=60) as resp:
            return json.loads(resp.read())['choices'][0]['message']['content']


In [ ]:
brain_a, brain_b, brain_c = BrainA(), BrainB(), BrainCClient()
session = {'candidate_id': 'sim_001', 'last_question': '', 'last_topic': '', 'last_level': 2, 'transcript': []}
context = 'Senior backend role requiring API reliability, Redis caching, deployment rollback, and metrics ownership.'
turns = [
    'I used Redis caching to reduce p95 latency from 900ms to 250ms.',
    'Maine cache lagaya tha kyunki database queries slow ho rahi thi.',
    'Deployment ke baad rollback plan ready tha, but metrics initially unstable the.'
]
latencies = []
for i, turn in enumerate(turns):
    start = time.perf_counter()
    behavior = brain_b.analyze(turn)
    ack_ms = (time.perf_counter() - start) * 1000
    if session['last_question']:
        raw = brain_c.chat([{'role': 'system', 'content': context}, {'role': 'user', 'content': wrap_evaluate(session['last_question'], turn)}])
        parsed_eval = parse_reply(raw)
    topic = ['Redis caching', 'API reliability', 'deployment rollback'][i]
    raw = brain_c.chat([{'role': 'system', 'content': context}, {'role': 'user', 'content': wrap_generate(topic, 2)}])
    parsed = parse_reply(raw)
    ttft_ms = (time.perf_counter() - start) * 1000
    latencies.append(ttft_ms)
    session['last_question'] = parsed['response']
    session['last_topic'] = topic
    brain_a.store(turn, behavior['language'])
    print({'mode': brain_c.mode, 'language': behavior['language'], 'ack': behavior['ack'], 'ack_ms': round(ack_ms, 2), 'ttft_ms': round(ttft_ms, 2), 'question': parsed['response']})
print('mean_ttft_ms', round(statistics.mean(latencies), 2))
